In [0]:
%pip install ddgs bs4 lxml requests -q

In [0]:
import hashlib
import uuid
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings("ignore", category=ResourceWarning)

# ==========================================
# 1. Smart Title & Company Extractor
# ==========================================
def extract_title_and_company(raw_title):
    # ATS సిస్టమ్స్ ఇచ్చే కామన్ చెత్త పదాలను తీసేస్తుంది
    clean_title = re.sub(r'(?i)Job Application for\s+', '', raw_title)
    clean_title = re.sub(r'(?i)Jobs In\s+', ' at ', clean_title)
    clean_title = re.sub(r'(?i)\s*-\s*Greenhouse.*$', '', clean_title)
    clean_title = re.sub(r'(?i)\s*-\s*Lever.*$', '', clean_title)
    clean_title = re.sub(r'(?i)\s*\|\s*LinkedIn.*$', '', clean_title)
    
    company = "Web Extracted"
    job_title = clean_title
    
    # ' at ', ' - ', ' | ' లాంటివి చూసి టైటిల్ ని కంపెనీని విడదీస్తుంది
    split_chars = [' at ', ' At ', ' - ', ' | ', ' @ ']
    for char in split_chars:
        if char in clean_title:
            parts = clean_title.split(char, 1)
            job_title = parts[0].strip()
            company = parts[1].strip()
            break
            
    # లాస్ట్ క్లీనప్ (Special characters తీసేయడానికి)
    job_title = re.sub(r'[^a-zA-Z0-9\s\+\#\.]', '', job_title).strip()
    company = re.sub(r'[^a-zA-Z0-9\s]', '', company).strip()
    
    return job_title.title(), company.title()

# ==========================================
# 2. Deep Link Scraper (With LinkedIn Bypass)
# ==========================================
def extract_real_job_data(url, snippet):
    # 💡 ANTI-CAPTCHA LOGIC: LinkedIn బాట్స్ ని బ్లాక్ చేస్తుంది కాబట్టి దాన్ని ఓపెన్ చేయము.
    # కేవలం స్నిప్పెట్ ని మాత్రమే వాడతాం.
    if "linkedin.com" in url:
        return "LinkedIn Job", snippet
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            for element in soup(["script", "style", "footer", "nav", "header"]):
                element.extract()
            page_title = soup.title.string if soup.title else ""
            text = soup.get_text(separator=' ', strip=True)
            clean_text = re.sub(r'\s+', ' ', text)
            return page_title, clean_text[:4500] # ఎగ్జాక్ట్ డిస్క్రిప్షన్!
    except Exception:
        pass
    return "", snippet

def process_search_result(res, role):
    apply_link = res.get('href', '')
    snippet = res.get('body', '')
    
    if not apply_link or "taptap" in apply_link.lower() or "zhihu" in apply_link.lower():
        return None 
        
    real_title, real_description = extract_real_job_data(apply_link, snippet)
    raw_title = real_title if real_title else res.get('title', '')
    
    # టైటిల్ ని కంపెనీని పర్ఫెక్ట్ గా లాగడం
    job_title, company = extract_title_and_company(raw_title)
    
    if not job_title or job_title == "":
        job_title = role
        
    job_hash = hashlib.md5(apply_link.encode('utf-8')).hexdigest()
    now = datetime.now()
    
    return {
        "id": str(uuid.uuid4()),
        "job_hash": job_hash,
        "created_at": now,
        "updated_at": None,
        "fetch_date": now.date(),
        "search_keyword": role,
        "company_name": company, 
        "job_title": job_title, 
        "job_description": real_description,
        "apply_link": apply_link,
        "validation_status": "Pending"
    }

# ==========================================
# 3. The Extreme Output Harvester Engine
# ==========================================
def execute_extreme_harvester(roles_list, locations_list):
    print("🔥 Initiating THE EXTREME HARVESTER (V4) Engine...")
    
    target_sites = [
        "site:greenhouse.io",
        "site:jobs.lever.co",
        "site:myworkdayjobs.com",
        "site:boards.ashbyhq.com",
        "site:linkedin.com/jobs/view" 
    ]
    
    locations_str = " OR ".join([f'"{loc}"' for loc in locations_list])
    records = []
    
    try:
        with DDGS() as ddgs:
            for role in roles_list:
                for site in target_sites:
                    query = f'{site} "{role}" ({locations_str})'
                    print(f"   🔍 Querying: {query}")
                    
                    try:
                        # 💡 max_results=20 పెట్టాం కాబట్టి ఒకేసారి చాలా జాబ్స్ వస్తాయి.
                        results = list(ddgs.text(query, backend='lite', max_results=20))
                        
                        if results:
                            print(f"      📥 Found {len(results)} links. Processing securely...")
                            
                            with ThreadPoolExecutor(max_workers=4) as executor:
                                futures = [executor.submit(process_search_result, res, role) for res in results]
                                for future in as_completed(futures):
                                    job_data = future.result()
                                    if job_data:
                                        records.append(job_data)
                                        
                        # సెర్చ్ కి సెర్చ్ కి మధ్య టైమ్ గ్యాప్ (యాంటీ-బాట్)
                        time.sleep(random.uniform(2.5, 4.5))
                    except Exception:
                        time.sleep(5) 
                        
    except Exception as e:
        print(f"❌ Fatal Web Scraping Error: {str(e)}")
        
    if not records:
        print("\n🛑 No jobs found. Try again in an hour.")
        return
        
    print(f"\n✅ EXTREME HARVEST COMPLETE! Grabbed {len(records)} clean links.")
    
    # ==========================================
    # 4. Delta Merge with New Schema
    # ==========================================
    # new_jobs_df = spark.createDataFrame(records)
    # ordered_columns = [
    #     "id", "job_hash", "created_at", "updated_at", "fetch_date", "search_keyword", 
    #     "company_name", "job_title", "job_description", "apply_link", "validation_status"
    # ]
    # new_jobs_df = new_jobs_df.select(*ordered_columns)

    # ==========================================
    # 4. Delta Merge with STRICT SCHEMA
    # ==========================================
    from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

    # 💡 THE FIX: Explicitly telling Spark the exact data types!
    job_schema = StructType([
        StructField("id", StringType(), True),
        StructField("job_hash", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True),
        StructField("fetch_date", DateType(), True),
        StructField("search_keyword", StringType(), True),
        StructField("company_name", StringType(), True),
        StructField("job_title", StringType(), True),
        StructField("job_description", StringType(), True),
        StructField("apply_link", StringType(), True),
        StructField("validation_status", StringType(), True)
    ])
    
    # Passing the strict schema here solves the CANNOT_DETERMINE_TYPE error
    new_jobs_df = spark.createDataFrame(records, schema=job_schema)
    
    # We don't need to select ordered_columns anymore because the schema strictly enforces the order!
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"🎉 100% Success! Clean, structured {len(records)} jobs merged into Staging without duplicates.")
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"🎉 100% Success! Clean, structured jobs merged into Staging without duplicates.")

# 🚀 RUN THE EXTREME ENGINE
execute_extreme_harvester(["Data Engineer", "Python Developer"], ["USA", "Remote", "Texas", "New York", "California"])

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging